In [10]:
import sys, os, importlib, importlib.util
from pathlib import Path
SRC = Path('/content/drive/MyDrive/CALSHIFT_Research/calshift-research/src')
f = SRC / 'config.py'
print('exists:', f.exists(), '| bytes:', f.stat().st_size if f.exists() else 0)

sys.modules.pop('config', None)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
importlib.invalidate_caches()

try:
    import config
    print('imported via sys.path')
except ModuleNotFoundError:
    spec = importlib.util.spec_from_file_location('config', f)
    config = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(config)
    sys.modules['config'] = config
    print('loaded directly from file')

import numpy as np, pandas as pd
os.chdir('/content/drive/MyDrive/CALSHIFT_Research/calshift-research')
print('ready:', os.getcwd(), '| seeds:', config.SEEDS, '| alpha:', config.ALPHA_PRIMARY)

exists: True | bytes: 1429
imported via sys.path
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research | seeds: [42, 1337, 2024, 7, 91, 512, 6021, 88, 3407, 12345] | alpha: 0.05


In [11]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, shutil, hashlib, subprocess, time
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
PARENT_DIR   = DRIVE_ROOT / 'CALSHIFT_Research'
PROJECT_ROOT = PARENT_DIR / 'calshift-research'
CRED_DIR     = DRIVE_ROOT / '.gitcreds'

subprocess.run(['git','config','--global','user.name','Md Anas Biswas'], check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'], check=False)
subprocess.run(['git','config','--global','credential.helper','store'], check=False)

for fn, dest in [('.git-credentials','/root/.git-credentials'),
                 ('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR / fn, CRED_DIR / fn):
        if cand.exists():
            shutil.copy(cand, dest); os.chmod(dest, 0o600); break

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
subprocess.run(['git','pull','--ff-only','--quiet'], check=False)

import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [12]:
# =============================================================================
# 04_models_and_calibration
# Cell 2 - decisions recorded BEFORE any fitting
# =============================================================================
DECISIONS = {
    'resampling': 'none',
    'class_weighting': 'none',
    'rationale': ('Mondrian conformal calibrates per class, so rare-class coverage is '
                  'controlled by the per-class quantile rather than by the classifier '
                  'prior. Resampling would add a design factor the preregistration '
                  'does not specify and would change the probability scale the '
                  'calibrator is fitted on.'),
    'hyperparameter_selection': 'once per architecture on D_val by macro-F1, then frozen',
    'probability_calibration': 'one-vs-rest isotonic on D_probcal, renormalised',
    'calibrator_source': 'source only, held fixed across all protocols and arms',
    'seeds_control': 'model training only; source partition fixed (nb02 PARTITION_SEED)',
}

GRID = {
    'rf':  {'n_estimators': [300], 'max_depth': [None, 20], 'min_samples_leaf': [1, 5]},
    'xgb': {'n_estimators': [300], 'max_depth': [6, 8], 'learning_rate': [0.1],
            'subsample': [0.8], 'colsample_bytree': [0.8]},   # required: without
            # subsampling XGBoost is deterministic and random_state has no effect,
            # which would give the (1|seed) variance component a structural zero
    'mlp': {'hidden_layer_sizes': [(128, 64)], 'alpha': [1e-4, 1e-3]},
}

(config.REPORTS_DIR / 'model_decisions.json').write_text(
    json.dumps({'decisions': DECISIONS, 'grid': {k: {kk: [str(x) for x in vv]
                for kk, vv in v.items()} for k, v in GRID.items()}}, indent=2))
print(json.dumps(DECISIONS, indent=2))
print('\ngrid recorded to reports/model_decisions.json')

{
  "resampling": "none",
  "class_weighting": "none",
  "rationale": "Mondrian conformal calibrates per class, so rare-class coverage is controlled by the per-class quantile rather than by the classifier prior. Resampling would add a design factor the preregistration does not specify and would change the probability scale the calibrator is fitted on.",
  "hyperparameter_selection": "once per architecture on D_val by macro-F1, then frozen",
  "probability_calibration": "one-vs-rest isotonic on D_probcal, renormalised",
  "calibrator_source": "source only, held fixed across all protocols and arms",
  "seeds_control": "model training only; source partition fixed (nb02 PARTITION_SEED)"
}

grid recorded to reports/model_decisions.json


In [13]:
# =============================================================================
# Cell 3 - write src/features.py (canonical encoder, shared by all notebooks)
# =============================================================================
FEATURES_PY = '\n'.join([
 '"""Canonical feature encoding for NSL-KDD. Single source of truth."""',
 'import numpy as np, pandas as pd',
 '',
 'CAT_COLS = ["protocol_type", "service", "flag"]',
 'DROP_COLS = ("label", "subtype", "partition", "is_unseen")',
 '',
 'def feature_cols(df):',
 '    return [c for c in df.columns if c not in DROP_COLS]',
 '',
 'def fit_categories(*frames):',
 '    cats = {}',
 '    for c in CAT_COLS:',
 '        vals = pd.concat([f[c].astype(str) for f in frames])',
 '        cats[c] = pd.Categorical(vals).categories',
 '    return cats',
 '',
 'def encode(df, cols, cats):',
 '    X = df[cols].copy()',
 '    for c in CAT_COLS:',
 '        X[c] = pd.Categorical(X[c].astype(str), categories=cats[c]).codes',
 '    return X.astype(np.float32).to_numpy()',
])
(config.PROJECT_ROOT / 'src' / 'features.py').write_text(FEATURES_PY + '\n')

import importlib, features
importlib.reload(features)
print('src/features.py written')

src/features.py written


In [14]:
# =============================================================================
# Cell 4 - load partitions and build matrices
# =============================================================================
nsl_train = pd.read_parquet(config.INTERIM_DIR / 'nslkdd_train.parquet').reset_index(drop=True)
nsl_test  = pd.read_parquet(config.INTERIM_DIR / 'nslkdd_test.parquet').reset_index(drop=True)
part = pd.read_parquet(config.PROC_DIR / 'nslkdd_source_partition_labels.parquet')
nsl_train = nsl_train.assign(partition=part['partition'].values)

fp = json.loads((config.REPORTS_DIR / 'partition_fingerprints.json').read_text())['fingerprints']
def _fp(idx):
    return hashlib.sha256(np.sort(np.asarray(idx, dtype=np.int64)).tobytes()).hexdigest()
for name in ['train', 'val', 'probcal', 'source_cal_pool']:
    got = _fp(nsl_train[nsl_train.partition == name].index)
    assert got == fp[f'source/{name}']['sha256'], f'{name} partition differs from nb02'
print('all four source partitions match the notebook 02 fingerprints')

D_train   = nsl_train[nsl_train.partition == 'train']
D_val     = nsl_train[nsl_train.partition == 'val']
D_probcal = nsl_train[nsl_train.partition == 'probcal']
S_pool    = nsl_train[nsl_train.partition == 'source_cal_pool']

COLS = features.feature_cols(nsl_train)
CATS = features.fit_categories(nsl_train, nsl_test)
CLASSES = config.CANONICAL_CLASSES
cls_to_i = {c: i for i, c in enumerate(CLASSES)}

def XY(df):
    return features.encode(df, COLS, CATS), df['label'].map(cls_to_i).to_numpy()

X_tr, y_tr = XY(D_train); X_va, y_va = XY(D_val)
X_pc, y_pc = XY(D_probcal); X_sp, y_sp = XY(S_pool)
X_te, y_te = XY(nsl_test)

print('train', X_tr.shape, '| val', X_va.shape, '| probcal', X_pc.shape,
      '| S_pool', X_sp.shape, '| target', X_te.shape)
print('train class counts:', dict(zip(CLASSES, np.bincount(y_tr, minlength=5))))

all four source partitions match the notebook 02 fingerprints
train (75590, 41) | val (12595, 41) | probcal (18894, 41) | S_pool (18894, 41) | target (22544, 41)
train class counts: {'Normal': np.int64(40407), 'DoS': np.int64(27557), 'Probe': np.int64(6995), 'R2L': np.int64(598), 'U2R': np.int64(33)}


In [15]:
# =============================================================================
# Cell 5 - hyperparameter selection on D_val, once per architecture, then frozen
# =============================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from itertools import product
import xgboost as xgb

SCALER = StandardScaler().fit(X_tr)
Xs_tr, Xs_va = SCALER.transform(X_tr), SCALER.transform(X_va)

def make(arch, params, seed):
    if arch == 'rf':
        return RandomForestClassifier(random_state=seed, n_jobs=-1, **params)
    if arch == 'xgb':
        return xgb.XGBClassifier(random_state=seed, n_jobs=-1, tree_method='hist',
                                 objective='multi:softprob', num_class=len(CLASSES),
                                 eval_metric='mlogloss', verbosity=0, **params)
    return MLPClassifier(random_state=seed, max_iter=120, early_stopping=True,
                         n_iter_no_change=8, **params)

SEL_SEED = 0
best = {}
t0 = time.time()
for arch, grid in GRID.items():
    keys = list(grid); results = []
    for vals in product(*[grid[k] for k in keys]):
        p = dict(zip(keys, vals))
        m = make(arch, p, SEL_SEED)
        if arch == 'mlp': m.fit(Xs_tr, y_tr); pred = m.predict(Xs_va)
        else:            m.fit(X_tr,  y_tr); pred = m.predict(X_va)
        f1 = f1_score(y_va, pred, average='macro')
        results.append((f1, p))
        print(f'  {arch:4s} {p} macro-F1 {f1:.4f}')
    results.sort(key=lambda r: -r[0])
    best[arch] = results[0][1]
    print(f'{arch}: selected {best[arch]} (macro-F1 {results[0][0]:.4f})\n')

(config.REPORTS_DIR / 'selected_hyperparameters.json').write_text(
    json.dumps({k: {kk: str(vv) for kk, vv in v.items()} for k, v in best.items()}, indent=2))
print(f'selection done in {time.time()-t0:.0f}s; FROZEN')

  rf   {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 1} macro-F1 0.9501
  rf   {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 5} macro-F1 0.8387
  rf   {'n_estimators': 300, 'max_depth': 20, 'min_samples_leaf': 1} macro-F1 0.9489
  rf   {'n_estimators': 300, 'max_depth': 20, 'min_samples_leaf': 5} macro-F1 0.8385
rf: selected {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 1} (macro-F1 0.9501)

  xgb  {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8} macro-F1 0.9266
  xgb  {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8} macro-F1 0.9267
xgb: selected {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8} (macro-F1 0.9267)

  mlp  {'hidden_layer_sizes': (128, 64), 'alpha': 0.0001} macro-F1 0.9056
  mlp  {'hidden_layer_sizes': (128, 64), 'alpha': 0.001} macro-F1 0.9264
mlp: selected {'hidden_la

In [16]:
# =============================================================================
# Cell 6 - isotonic probability calibration, fitted on D_probcal (source only)
# =============================================================================
from sklearn.isotonic import IsotonicRegression

def fit_calibrator(p_raw, y, n_classes):
    isos = []
    for k in range(n_classes):
        iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
        iso.fit(p_raw[:, k], (y == k).astype(float))
        isos.append(iso)
    return isos

def apply_calibrator(isos, p_raw, eps=1e-12):
    P = np.column_stack([iso.predict(p_raw[:, k]) for k, iso in enumerate(isos)])
    P = np.clip(P, eps, None)
    return P / P.sum(axis=1, keepdims=True)     # renormalised (preregistration 6)

def ece_equal_mass(p_true_class, correct, n_bins=15):
    """Equal-mass binning; avoids the equal-width bias documented by Roelofs 2022."""
    order = np.argsort(p_true_class)
    p, c = p_true_class[order], correct[order]
    bins = np.array_split(np.arange(len(p)), n_bins)
    return float(sum(len(b) / len(p) * abs(c[b].mean() - p[b].mean())
                     for b in bins if len(b)))

print('calibration helpers ready')

calibration helpers ready


In [17]:
# =============================================================================
# Cell 7 - train 3 architectures x 10 seeds, calibrate, cache probabilities
# Cached: S_pool (for SHC) and the FULL target pool (any ladder instance indexes in).
# =============================================================================
config.PROC_DIR.mkdir(parents=True, exist_ok=True)
rows = []
t0 = time.time()

for arch in ['rf', 'xgb', 'mlp']:
    for seed in config.SEEDS:
        tag = f'{arch}_s{seed}'
        out = config.PROC_DIR / f'probs_{tag}.npz'
        if out.exists():
            print(f'  {tag} cached, skipping'); continue

        m = make(arch, best[arch], seed)
        if arch == 'mlp':
            m.fit(Xs_tr, y_tr)
            raw = lambda X: m.predict_proba(SCALER.transform(X))
        else:
            m.fit(X_tr, y_tr)
            raw = lambda X: m.predict_proba(X)

        isos = fit_calibrator(raw(X_pc), y_pc, len(CLASSES))
        P_sp = apply_calibrator(isos, raw(X_sp))
        P_te = apply_calibrator(isos, raw(X_te))
        P_pc = apply_calibrator(isos, raw(X_pc))
        np.savez_compressed(out, S_pool=P_sp.astype(np.float32),
                            target=P_te.astype(np.float32), probcal=P_pc.astype(np.float32))

        pr_raw = raw(X_te)
        pred_raw = pr_raw.argmax(1); pred_cal = P_te.argmax(1)
        corr_raw = (pred_raw == y_te).astype(float)
        corr_cal = (pred_cal == y_te).astype(float)
        # each ECE must be paired with the argmax that produced it
        e_before = ece_equal_mass(pr_raw.max(1), corr_raw)
        e_after  = ece_equal_mass(P_te.max(1),   corr_cal)
        rows.append({'arch': arch, 'seed': seed,
                     'macro_f1_raw': f1_score(y_te, pred_raw, average='macro'),
                     'macro_f1_cal': f1_score(y_te, pred_cal, average='macro'),
                     'argmax_changed': float((pred_raw != pred_cal).mean()),
                     'ece_before': e_before, 'ece_after': e_after})
        print(f'  {tag:10s} F1raw {rows[-1]["macro_f1_raw"]:.4f} '
              f'F1cal {rows[-1]["macro_f1_cal"]:.4f} | '
              f'ECE {e_before:.4f} -> {e_after:.4f} | '
              f'argmax changed {rows[-1]["argmax_changed"]:.3f} | {time.time()-t0:.0f}s')

perf = pd.DataFrame(rows)
if len(perf):
    perf.to_csv(config.REPORTS_DIR / 'model_performance.csv', index=False)
    print('\n', perf.groupby('arch')[['macro_f1_raw','macro_f1_cal','ece_before',
          'ece_after','argmax_changed']].agg(['mean','std']).round(4).to_string())
    zero_var = perf.groupby('arch')['macro_f1_raw'].std().fillna(0)
    dead = zero_var[zero_var < 1e-9].index.tolist()
    if dead:
        print('\nWARNING: zero seed variance for', dead,
              '- the (1|seed) term is structurally zero for these architectures')

  rf_s42     F1raw 0.4991 F1cal 0.5354 | ECE 0.1772 -> 0.2166 | argmax changed 0.016 | 5s
  rf_s1337   F1raw 0.5080 F1cal 0.5337 | ECE 0.1707 -> 0.2139 | argmax changed 0.017 | 10s
  rf_s2024   F1raw 0.5048 F1cal 0.5274 | ECE 0.1753 -> 0.2173 | argmax changed 0.016 | 16s
  rf_s7      F1raw 0.5061 F1cal 0.5407 | ECE 0.1722 -> 0.2116 | argmax changed 0.013 | 21s
  rf_s91     F1raw 0.5087 F1cal 0.5421 | ECE 0.1730 -> 0.2052 | argmax changed 0.019 | 26s
  rf_s512    F1raw 0.5050 F1cal 0.5199 | ECE 0.1771 -> 0.2170 | argmax changed 0.015 | 32s
  rf_s6021   F1raw 0.5010 F1cal 0.5433 | ECE 0.1748 -> 0.2107 | argmax changed 0.022 | 37s
  rf_s88     F1raw 0.5071 F1cal 0.5344 | ECE 0.1750 -> 0.2139 | argmax changed 0.019 | 42s
  rf_s3407   F1raw 0.5084 F1cal 0.5373 | ECE 0.1706 -> 0.2163 | argmax changed 0.011 | 48s
  rf_s12345  F1raw 0.4978 F1cal 0.5337 | ECE 0.1750 -> 0.2199 | argmax changed 0.021 | 53s
  xgb_s42    F1raw 0.5340 F1cal 0.5341 | ECE 0.2133 -> 0.2154 | argmax changed 0.003 | 60s


In [18]:
# =============================================================================
# Cell 8 - integrity checks on the cached probabilities
# =============================================================================
import glob as _g
files = sorted(_g.glob(str(config.PROC_DIR / 'probs_*.npz')))
print('cached model files:', len(files), '(expect 30)')

bad = []
for f in files:
    z = np.load(f)
    for key, n in [('S_pool', len(X_sp)), ('target', len(X_te)), ('probcal', len(X_pc))]:
        P = z[key]
        if P.shape != (n, len(CLASSES)): bad.append((f, key, 'shape', P.shape))
        if not np.allclose(P.sum(1), 1.0, atol=1e-4): bad.append((f, key, 'not normalised', None))
        if (P < 0).any(): bad.append((f, key, 'negative', None))
print('integrity problems:', bad if bad else 'none')
assert not bad

z = np.load(files[0])
print('\nexample calibrated row:', np.round(z['target'][0], 4), 'sum',
      round(float(z['target'][0].sum()), 6))

cached model files: 30 (expect 30)
integrity problems: none

example calibrated row: [0.000e+00 9.999e-01 0.000e+00 1.000e-04 0.000e+00] sum 1.0


In [19]:
# =============================================================================
# Cell 9 - manifest
# =============================================================================
manifest = {
    'n_models': len(files),
    'architectures': ['rf', 'xgb', 'mlp'],
    'seeds': config.SEEDS,
    'selected_hyperparameters': {k: {kk: str(vv) for kk, vv in v.items()}
                                 for k, v in best.items()},
    'decisions': DECISIONS,
    'cached_arrays': ['S_pool', 'target', 'probcal'],
    'calibrator': 'one-vs-rest isotonic on D_probcal, renormalised',
    'ece_estimator': 'equal-mass, 15 bins',
}
(config.REPORTS_DIR / 'models_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps({k: v for k, v in manifest.items() if k != 'decisions'}, indent=2))

{
  "n_models": 30,
  "architectures": [
    "rf",
    "xgb",
    "mlp"
  ],
  "seeds": [
    42,
    1337,
    2024,
    7,
    91,
    512,
    6021,
    88,
    3407,
    12345
  ],
  "selected_hyperparameters": {
    "rf": {
      "n_estimators": "300",
      "max_depth": "None",
      "min_samples_leaf": "1"
    },
    "xgb": {
      "n_estimators": "300",
      "max_depth": "8",
      "learning_rate": "0.1",
      "subsample": "0.8",
      "colsample_bytree": "0.8"
    },
    "mlp": {
      "hidden_layer_sizes": "(128, 64)",
      "alpha": "0.001"
    }
  },
  "cached_arrays": [
    "S_pool",
    "target",
    "probcal"
  ],
  "calibrator": "one-vs-rest isotonic on D_probcal, renormalised",
  "ece_estimator": "equal-mass, 15 bins"
}


In [20]:
def git(*args, show=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if show:
        if r.stdout.strip(): print(r.stdout.strip())
        if r.stderr.strip(): print(r.stderr.strip())
    return r

for s, d in [('/root/.git-credentials', PARENT_DIR / '.git-credentials'),
             ('/root/.gitconfig',       PARENT_DIR / '.gitconfig')]:
    if os.path.exists(s): shutil.copy(s, d)

os.chdir(PROJECT_ROOT)
git('add','-A', show=False)
if git('status','--porcelain', show=False).stdout.strip():
    git('commit','-m','nb04: models and calibration')
    r = git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else:
    print('nothing to commit')
print(git('log','--oneline','-3', show=False).stdout)

[main 0e3e6d4] nb04: models and calibration
 6 files changed, 178 insertions(+), 595 deletions(-)
 rewrite notebooks/04_models_and_calibration.ipynb (99%)
 create mode 100644 reports/model_decisions.json
 create mode 100644 reports/model_performance.csv
 create mode 100644 reports/models_manifest.json
 create mode 100644 reports/selected_hyperparameters.json
 create mode 100644 src/features.py
Branch 'main' set up to track remote branch 'main' from 'origin'.
To https://github.com/anasbiswas1/calshift-research.git
   d8822cf..0e3e6d4  main -> main
0e3e6d4 nb04: models and calibration
2ba000f revert to inline bootstrap, remove bootstrap module
d8822cf extract bootstrap to src, thin notebook cells

